# Day 5 — HOL: Build the Bronze Layer — All Sources
### GlobalMart Data Engineering · 3:00 PM – 5:30 PM

---

## What We Are Building

```
SOURCE                        BRONZE TABLE
──────────────────────────────────────────────────────────────────
ADLS raw/orders.csv        →  bronze/adls/orders          (Auto Loader)
ADLS raw/customers.csv     →  bronze/adls/customers       (Auto Loader)
ADLS raw/products.csv      →  bronze/adls/products        (Auto Loader)
ADLS raw/shipping_tier.csv →  bronze/adls/shipping_tiers  (Auto Loader)

Supabase orders            →  bronze/supabase/orders       (JDBC)
Supabase order_items       →  bronze/supabase/order_items  (JDBC)
──────────────────────────────────────────────────────────────────
```

## By the End of This HOL You Will Have

- All 6 Bronze Delta tables loaded and queryable
- Audit columns on every table (`_ingested_at`, `_source_system`, `_batch_id`, `_load_type`)
- Change Data Feed (CDF) enabled — ready for Silver incremental reads
- A watermark table — so JDBC incremental loads know exactly where to resume
- A foundation that supports CDC, SCD Type 1, and SCD Type 2 in future sessions

---
## Bronze Design Principles — Why We Build It This Way

### Rule 1 — Append Only, Never Overwrite

```
Bronze is a permanent raw record of everything that arrived.

Run 1 (Monday):   1,000 orders land → Bronze has 1,000 rows
Run 2 (Tuesday):  200 new orders  → Bronze has 1,200 rows  (NOT replaced)
Run 3 (Wednesday): 50 updates in source → Bronze has 1,250 rows (raw event captured)

Silver will deduplicate and apply SCD logic.
Bronze never does — it just absorbs everything.
```

### Rule 2 — Audit Columns on Every Table

| Column | ADLS Value | Supabase Value | Purpose |
|--------|-----------|----------------|---------|
| `_ingested_at` | `current_timestamp()` | `current_timestamp()` | When this row landed in Bronze |
| `_source_system` | `adls` | `supabase` | Which system sent this row |
| `_source_file` | full ADLS path | `null` | Which file (ADLS only) |
| `_batch_id` | UUID per run | UUID per run | Group all rows from one pipeline run |
| `_load_type` | `autoloader` | `full` / `incremental` | How this row was loaded |

### Rule 3 — Change Data Feed (CDF) Enabled

```
delta.enableChangeDataFeed = true

This records every INSERT/UPDATE/DELETE to the Bronze table in a change log.
Silver will read this log to process ONLY new changes — not re-scan the full table.
This is the foundation for CDC, SCD Type 1, and SCD Type 2.
```

### Rule 4 — Watermark Table for JDBC Sources

```
A small Delta table stores the last ingested timestamp per source:

  source_name          | last_watermark           | last_run_at
  supabase_orders      | 2026-06-18 10:00:00 UTC  | 2026-06-18 10:01:00 UTC
  supabase_order_items | 2026-06-18 10:00:00 UTC  | 2026-06-18 10:01:00 UTC

First run  → full load (no watermark exists yet)
All future runs → WHERE updated_at > last_watermark  (incremental only)
```

In [ ]:

# ─── SETUP ─────────────────────────────────────────────────────────────────────
# Unity Catalog external location configured — no storage key needed.
# Replace YOUR_SUPABASE_PASSWORD before running Part 2.

import uuid
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import StructType
from delta.tables import DeltaTable

# ── ADLS paths ────────────────────────────────────────────────────────────────
storage_account = "amazonprojectadls"
container       = "amazon-data"
base_path       = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

raw_base        = f"{base_path}/raw"
bronze_base     = f"{base_path}/bronze"
schema_base     = f"{base_path}/_autoloader/schemas/bronze"
checkpoint_base = f"{base_path}/_autoloader/checkpoints/bronze"

bronze_adls_orders         = f"{bronze_base}/adls/orders"
bronze_adls_customers      = f"{bronze_base}/adls/customers"
bronze_adls_products       = f"{bronze_base}/adls/products"
bronze_adls_shipping_tiers = f"{bronze_base}/adls/shipping_tiers"

bronze_pg_orders      = f"{bronze_base}/supabase/orders_new"
bronze_pg_order_items = f"{bronze_base}/supabase/order_items_new"

watermark_path = f"{bronze_base}/metadata/watermarks"

# ── Supabase / PostgreSQL credentials ─────────────────────────────────────────
# WARNING: Do NOT push this notebook to GitHub with the real password
# Project: amazondata_superbase (qcdequqeczifppyypzhs) — Sydney region
pg_host     = "db.qcdequqeczifppyypzhs.supabase.co"
pg_database = "postgres"
pg_user     = "postgres"
pg_password = "YOUR_SUPABASE_PASSWORD"    # ← paste your password here
jdbc_url    = f"jdbc:postgresql://{pg_host}:5432/{pg_database}?sslmode=require"

# ── Batch ID — unique per pipeline run ────────────────────────────────────────
batch_id = str(uuid.uuid4())[:8]

print(f"Base path : {base_path}")
print(f"PG host   : {pg_host}")
print(f"Batch ID  : {batch_id}")
print()
print("Bronze paths:")
for name, path in [
    ("adls/orders",          bronze_adls_orders),
    ("adls/customers",       bronze_adls_customers),
    ("adls/products",        bronze_adls_products),
    ("adls/shipping_tiers",  bronze_adls_shipping_tiers),
    ("supabase/orders_new",      bronze_pg_orders),
    ("supabase/order_items_new", bronze_pg_order_items),
]:
    print(f"  {name:<28} {path}")


In [ ]:
# ─── Helper: clean column names ────────────────────────────────────────────────
# Removes special characters that Delta doesn't allow in column names.
# Applied consistently to ALL sources so column names are predictable downstream.

def clean_columns(df):
    for c in df.columns:
        clean = (
            c.replace(" ", "_")
             .replace("(", "")
             .replace(")", "")
             .replace(".", "")
             .replace("%", "Pct")
             .replace("-", "_")
        )
        if clean != c:
            df = df.withColumnRenamed(c, clean)
    return df

print("clean_columns() helper loaded.")

---
## Part 1 — ADLS → Bronze (Auto Loader)

Auto Loader reads files from `raw/` incrementally.
The checkpoint tracks which files have been processed — so re-running this notebook
will only pick up NEW files, never re-process old ones.

```
ADLS raw/orders.csv          ─┐
ADLS raw/orders_20260619.csv  ├──► Auto Loader ──► bronze/adls/orders  (Delta)
ADLS raw/orders_20260620.csv  ┘
         ↑                              ↑
  new files land here           checkpoint tracks which
  (producer drops them)         files have been ingested
```

In [ ]:

# ─── Auto Loader ingestion function ────────────────────────────────────────────
# One function handles all ADLS sources — same pattern, different paths.
# Unity Catalog requires _metadata.file_path instead of input_file_name()

def ingest_adls_to_bronze(source_name, raw_path, bronze_path):
    print(f"Ingesting {source_name} ...")

    schema_path     = f"{schema_base}/{source_name}"
    checkpoint_path = f"{checkpoint_base}/{source_name}"

    raw_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",              "csv")
        .option("cloudFiles.schemaLocation",      schema_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header",      "true")
        .option("inferSchema", "true")
        .load(raw_path)
        .transform(clean_columns)
        .withColumn("_ingested_at",   current_timestamp())
        .withColumn("_source_file",   col("_metadata.file_path"))   # UC-compatible
        .withColumn("_source_system", lit("adls"))
        .withColumn("_batch_id",      lit(batch_id))
        .withColumn("_load_type",     lit("autoloader"))
    )

    (
        raw_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(bronze_path)
        .awaitTermination()
    )

    count = spark.read.format("delta").load(bronze_path).count()
    print(f"  Done. Bronze {source_name}: {count:,} total rows")

print("ingest_adls_to_bronze() loaded.")


In [ ]:
# ─── Ingest ADLS → Bronze: orders ──────────────────────────────────────────────

ingest_adls_to_bronze(
    source_name = "orders",
    raw_path    = f"{raw_base}/",
    bronze_path = bronze_adls_orders
)

spark.read.format("delta").load(bronze_adls_orders).show(3, truncate=False)

In [ ]:
# ─── Ingest ADLS → Bronze: customers ───────────────────────────────────────────

ingest_adls_to_bronze(
    source_name = "customers",
    raw_path    = f"{raw_base}/",
    bronze_path = bronze_adls_customers
)

spark.read.format("delta").load(bronze_adls_customers).show(3, truncate=False)

In [ ]:
# ─── Ingest ADLS → Bronze: products ────────────────────────────────────────────

ingest_adls_to_bronze(
    source_name = "products",
    raw_path    = f"{raw_base}/",
    bronze_path = bronze_adls_products
)

spark.read.format("delta").load(bronze_adls_products).show(3, truncate=False)

In [ ]:
# ─── Ingest ADLS → Bronze: shipping_tiers ──────────────────────────────────────
# shipping_tier.csv has column 'Cost (Rs)' — clean_columns() handles it.

ingest_adls_to_bronze(
    source_name = "shipping_tiers",
    raw_path    = f"{raw_base}/",
    bronze_path = bronze_adls_shipping_tiers
)

spark.read.format("delta").load(bronze_adls_shipping_tiers).show(truncate=False)

---
## Incremental File Detection — How It Works in Production

Auto Loader does NOT re-scan the full folder on every run.
It keeps a **checkpoint** that records which files have already been processed.

```
Run 1 (today):
  raw/ has:  orders.csv
  Checkpoint records: orders.csv ✓ processed
  Bronze gets: 1,463,800 rows

A supplier drops orders_day2.csv into raw/
  (uploaded via Azure Portal / ADF / any external process — NOT from this notebook)

Run 2 (next trigger):
  raw/ has:  orders.csv   orders_day2.csv
  Checkpoint says: orders.csv already done — SKIP IT
  Auto Loader processes: orders_day2.csv ONLY
  Bronze gets: 1,463,800 + N new rows   ← only the delta
```

### The Checkpoint Is the Key

```
_autoloader/checkpoints/bronze/orders/
    commits/
    offsets/
    sources/0/
        _metadata/  ← list of every file Auto Loader has seen
```

As long as the checkpoint folder exists, Auto Loader remembers every file it has ingested.
Re-running = only new files processed. Old files skipped automatically. No duplicates.

---

### How to Drop a New File (Upload from OUTSIDE Databricks)

**Option 1 — Azure Portal (for this HOL):**
```
Azure Portal
  → Storage Accounts → amazonprojectadls
  → Containers → amazon-data → raw/
  → Upload → pick your new CSV file (e.g. orders_day2.csv)
```

**Option 2 — Azure Storage Explorer (desktop tool):**
```
Connect to amazonprojectadls → amazon-data → raw/
Drag and drop the new CSV file
```

**Option 3 — Production (fully automated):**
```
Supplier FTP / API → Azure Data Factory pipeline → drops file into raw/
Event Grid fires notification → Auto Loader picks it up within seconds
No human action needed — ever
```

> **Upload a new CSV file to `raw/` in ADLS now using Azure Portal, then run the next two cells.**

In [ ]:
# ─── Before: snapshot current Bronze state ─────────────────────────────────────
# Run this BEFORE uploading the new file — records the baseline.

before_df    = spark.read.format("delta").load(bronze_adls_orders)
before_count = before_df.count()

print(f"Bronze orders — row count before new file: {before_count:,}")
print()
print("Files already ingested (each row = one file Auto Loader has seen):")
(
    before_df
    .select("_source_file", "_ingested_at")
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
    .show(truncate=False)
)
print("─" * 60)
print("Now upload a new CSV to raw/ in Azure Portal, then run the next cell.")

In [ ]:
# ─── After: re-trigger Auto Loader — only the NEW file gets processed ───────────
# The file was uploaded externally (Azure Portal / ADF / supplier).
# Nothing in this notebook wrote the file.
# Auto Loader reads the checkpoint → skips old files → processes only the new one.

import uuid
batch_id = str(uuid.uuid4())[:8]   # fresh batch ID for this incremental run

ingest_adls_to_bronze(
    source_name = "orders",
    raw_path    = f"{raw_base}/",
    bronze_path = bronze_adls_orders
)

after_count = spark.read.format("delta").load(bronze_adls_orders).count()
new_rows    = after_count - before_count

print()
print(f"  Before : {before_count:,} rows")
print(f"  After  : {after_count:,} rows")
print(f"  New    : {new_rows:,} rows  ← came from the new file only")
print()
print("Breakdown by file — new file should appear with its own row count:")
(
    spark.read.format("delta").load(bronze_adls_orders)
    .select("_source_file", "_batch_id", "_ingested_at")
    .groupBy("_source_file", "_batch_id")
    .count()
    .orderBy("_ingested_at")
    .show(truncate=False)
)

---
## Part 2 — PostgreSQL (Supabase) → Bronze (JDBC)

JDBC does not have a built-in checkpoint like Auto Loader.
We manage incremental loading ourselves using a **watermark table**.

```
FIRST RUN:
  1. No watermark exists → full load (SELECT * FROM orders)
  2. Save max(updated_at) as the watermark
  3. Append all rows to Bronze with _load_type = 'full'

EVERY SUBSEQUENT RUN:
  1. Read watermark from Delta table
  2. Query only new/changed rows: SELECT * WHERE updated_at > last_watermark
  3. Append to Bronze with _load_type = 'incremental'
  4. Update watermark to new max(updated_at)
```

### Why `updated_at` as the watermark column?

```
updated_at is set by the source system on every INSERT and UPDATE.
  → New orders:    updated_at = order creation time
  → Updated orders: updated_at = time of change

Limitation: does NOT capture hard DELETEs (row removed from source).
  → CDC will handle deletes later (Debezium / Supabase CDC)
  → For now: watermark captures inserts + updates reliably
```

In [ ]:
# ─── Verify JDBC connection ─────────────────────────────────────────────────────

def jdbc_read(query_or_table):
    return (
        spark.read
        .format("jdbc")
        .option("url",      jdbc_url)
        .option("dbtable",  query_or_table)
        .option("user",     pg_user)
        .option("password", pg_password)
        .option("driver",   "org.postgresql.Driver")
        .load()
    )

# Quick connection test — queries orders_new to confirm table exists and is accessible
test_df = jdbc_read("(SELECT COUNT(*) AS total_orders FROM orders_new) t")
test_df.show()
print("JDBC connection OK.")

In [ ]:
# ─── Watermark table helpers ────────────────────────────────────────────────────

from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import max as spark_max
from datetime import datetime, timezone

EPOCH = "1970-01-01 00:00:00"   # default watermark for first run

def get_watermark(source_name):
    """Returns last watermark timestamp string for this source, or EPOCH if none."""
    try:
        wm_df = spark.read.format("delta").load(watermark_path)
        rows  = wm_df.filter(f"source_name = '{source_name}'").collect()
        if rows:
            return str(rows[0]["last_watermark"])
    except Exception:
        pass  # watermark table doesn't exist yet
    return EPOCH


def save_watermark(source_name, new_watermark):
    """Upserts the watermark for this source into the watermark Delta table."""
    schema = StructType([
        StructField("source_name",     StringType(),    False),
        StructField("last_watermark",  StringType(),    False),
        StructField("last_updated_at", StringType(),    False),
    ])
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    new_row = spark.createDataFrame(
        [(source_name, str(new_watermark), now)], schema
    )

    try:
        wm_table = DeltaTable.forPath(spark, watermark_path)
        (
            wm_table.alias("wm")
            .merge(
                new_row.alias("new"),
                "wm.source_name = new.source_name"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    except Exception:
        new_row.write.format("delta").mode("overwrite").save(watermark_path)

    print(f"  Watermark saved: {source_name} → {new_watermark}")


print("Watermark helpers loaded.")

In [ ]:
# ─── JDBC ingestion function (full load first run, incremental after) ───────────

def ingest_jdbc_to_bronze(source_name, table_name, watermark_col, bronze_path):
    print(f"Ingesting {source_name} from Supabase ...")

    last_wm   = get_watermark(source_name)
    load_type = "full" if last_wm == EPOCH else "incremental"

    print(f"  Load type  : {load_type}")
    print(f"  Watermark  : {last_wm}")

    if load_type == "full":
        query = f"(SELECT * FROM {table_name}) t"
    else:
        query = f"(SELECT * FROM {table_name} WHERE {watermark_col} > '{last_wm}') t"

    df = (
        jdbc_read(query)
        .transform(clean_columns)
        .withColumn("_ingested_at",   current_timestamp())
        .withColumn("_source_system", lit("supabase"))
        .withColumn("_source_file",   lit(None).cast("string"))
        .withColumn("_batch_id",      lit(batch_id))
        .withColumn("_load_type",     lit(load_type))
    )

    new_rows = df.count()

    if new_rows == 0:
        print(f"  No new rows since last watermark. Skipping write.")
        return

    df.write.format("delta").mode("append").option("mergeSchema", "true").save(bronze_path)

    # Update watermark to the latest value in what we just loaded
    new_wm = df.agg(spark_max(col(watermark_col))).collect()[0][0]
    save_watermark(source_name, new_wm)

    total = spark.read.format("delta").load(bronze_path).count()
    print(f"  Loaded {new_rows:,} new rows. Bronze {source_name}: {total:,} total rows")


print("ingest_jdbc_to_bronze() loaded.")

In [ ]:

# ─── Ingest Supabase → Bronze: orders_new ──────────────────────────────────────
# First run: full load | Subsequent runs: only rows where updated_at > last watermark

ingest_jdbc_to_bronze(
    source_name   = "supabase_orders",
    table_name    = "orders_new",
    watermark_col = "updated_at",
    bronze_path   = bronze_pg_orders
)

spark.read.format("delta").load(bronze_pg_orders).show(3, truncate=False)


In [ ]:

# ─── Ingest Supabase → Bronze: order_items_new ─────────────────────────────────

ingest_jdbc_to_bronze(
    source_name   = "supabase_order_items",
    table_name    = "order_items_new",
    watermark_col = "updated_at",
    bronze_path   = bronze_pg_order_items
)

spark.read.format("delta").load(bronze_pg_order_items).show(3, truncate=False)


---
## Part 3 — Prepare Bronze for CDC / CDF / SCD

Now that all 6 tables are loaded, enable **Change Data Feed** on each one.

```
delta.enableChangeDataFeed = true

This tells Delta to maintain a change log alongside the table data.
Every INSERT, UPDATE, DELETE is recorded with:
  _change_type:         insert | update_preimage | update_postimage | delete
  _commit_version:      Delta table version number
  _commit_timestamp:    when the change happened

Silver will read:
  spark.read.format("delta")
       .option("readChangeFeed", "true")
       .option("startingVersion", last_processed_version)
       .load(bronze_path)

This gives Silver ONLY the rows that changed since last run — not the full table.
That is the foundation for SCD Type 1 and SCD Type 2.
```

In [ ]:
# ─── Enable Change Data Feed on all Bronze tables ───────────────────────────────

bronze_tables = {
    "adls_orders":          bronze_adls_orders,
    "adls_customers":       bronze_adls_customers,
    "adls_products":        bronze_adls_products,
    "adls_shipping_tiers":  bronze_adls_shipping_tiers,
    "supabase_orders":      bronze_pg_orders,
    "supabase_order_items": bronze_pg_order_items,
}

for name, path in bronze_tables.items():
    spark.sql(f"""
        ALTER TABLE delta.`{path}`
        SET TBLPROPERTIES (
            'delta.enableChangeDataFeed'         = 'true',
            'delta.autoOptimize.optimizeWrite'   = 'true',
            'delta.autoOptimize.autoCompact'     = 'true'
        )
    """)
    print(f"  CDF + AUTO OPTIMIZE enabled: {name}")

print()
print("All Bronze tables ready for incremental Silver processing.")

In [ ]:
# ─── Verify: row counts across all Bronze tables ────────────────────────────────

print(f"{'TABLE':<30} {'ROWS':>10}  PATH")
print("-" * 90)

for name, path in bronze_tables.items():
    try:
        count = spark.read.format("delta").load(path).count()
        print(f"  {name:<28} {count:>10,}  {path}")
    except Exception as e:
        print(f"  {name:<28} {'ERROR':>10}  {str(e)[:60]}")

In [ ]:
# ─── Verify: watermark table ────────────────────────────────────────────────────

print("Watermark table — current state:")
spark.read.format("delta").load(watermark_path).show(truncate=False)

In [ ]:
# ─── Verify: CDF properties on Bronze orders ────────────────────────────────────

spark.sql(f"DESCRIBE DETAIL delta.`{bronze_pg_orders}`").select(
    "name", "numFiles", "sizeInBytes", "numRows"
).show(truncate=False)

spark.sql(f"""
    SHOW TBLPROPERTIES delta.`{bronze_pg_orders}`
""").filter("key LIKE 'delta%'").show(truncate=False)

---
## What We Have Now vs What's Coming Next

```
BUILT TODAY — Bronze Layer
─────────────────────────────────────────────────────────────────────────
  bronze/adls/orders          ← all ADLS orders files (Auto Loader)
  bronze/adls/customers       ← all ADLS customers files
  bronze/adls/products        ← all ADLS product catalog files
  bronze/adls/shipping_tiers  ← all ADLS shipping tier files
  bronze/supabase/orders      ← full load from PostgreSQL, watermarked
  bronze/supabase/order_items ← full load from PostgreSQL, watermarked
  bronze/metadata/watermarks  ← tracks last ingested timestamp per source

  All tables: append-only, CDF enabled, AUTO OPTIMIZE on

NEXT SESSIONS — Silver Layer (uses Bronze as input)
─────────────────────────────────────────────────────────────────────────
  CDC (Week 2):
    Supabase emits change events → Bronze captures them with _op column
    Silver reads CDF log, applies MERGE to deduplicate

  SCD Type 1 (overwrite):
    Silver reads CDF → MERGE into silver table → overwrites changed rows
    No history kept — always shows current state

  SCD Type 2 (history):
    Silver reads CDF → for each change, closes old row (is_current=false)
    and inserts new row (is_current=true, valid_from, valid_to)
    Full history of every customer/product change preserved
```

---

## Discussion Questions

1. *Why do we append to Bronze instead of overwriting? When would overwriting cause a problem?*

2. *The watermark uses `updated_at`. What type of change in Supabase would the watermark NOT capture? How would CDC fix this?*

3. *What does `_load_type = incremental` in Bronze tell the Silver pipeline that `_load_type = full` does not?*

4. *Why do we enable CDF on Bronze and not just re-scan the whole table every time Silver runs?*

5. *A new column `loyalty_tier` is added to the `orders` table in Supabase. What happens the next time `ingest_jdbc_to_bronze()` runs? Does anything break?*